# TSAI Exercise Sheet 8

In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from random import randint
from tqdm import trange

## Dataset

In [ ]:
# No need to change anything here
class TimeSeriesDataset():
    def __init__(self, data, sequence_length=200, batch_size=16):
        self.X = torch.tensor(data, dtype=torch.float32)
        self.total_time_steps = self.X.shape[0]
        self.sequence_length = sequence_length
        self.batch_size = batch_size

    def __len__(self):
        return self.total_time_steps - self.sequence_length - 1

    def __getitem__(self, t):
        x = self.X[t:t+self.sequence_length, :]
        y = self.X[t+1:t+self.sequence_length+1, :]
        return x, y

    def sample_batch(self):
        X = []
        Y = []
        for _ in range(self.batch_size):
            idx = randint(0, len(self))
            x, y = self[idx]
            X.append(x)
            Y.append(y)

        return torch.stack(X), torch.stack(Y)

## Models

In [2]:
class PLRNN_Base(nn.Module):
    def __init__(self, M, P, N):
        super(PLRNN_Base, self).__init__()

        # Initialize model dimensions
        self.M = M # latent dimension
        self.P = P # number of ReLUs
        self.N = N # readout dimension

        # Initialize model parameters A, W, h, B    
        self.A, self.W, self.h = self.initialize_AWh_random()
        self.B = self.init_uniform((self.N,self.M))
    
    def forward(self, z):
        raise NotImplementedError("The forward method is not yet implemented.")
    
    def initialize_AWh_random(self):
        #Randomly initialize A, W, h
        A = nn.Parameter(torch.diagonal(self.normalized_positive_definite(self.M),0)) # Create diagonal matrix A from normalized positive definite matrix
        W = nn.Parameter(torch.randn(self.M, self.M)*0.01) # Initialize weight matrix W with gaussian random numbers
        h = nn.Parameter(torch.zeros(self.M)) # Initialize bias vector h to zero
        return A, W, h
    
    def normalized_positive_definite(self,M):
        # Generate a normalized positive definite matrix
        R = np.random.randn(M, M).astype(np.float32)
        K = np.matmul(R.T, R) / M + np.eye(M)  # R'R ./ M + I
        eigenvalues = np.linalg.eigvals(K)
        lambda_max = np.max(np.abs(eigenvalues))
        return torch.tensor(K / lambda_max).float()
    
    def init_uniform(self, shape):
        # Initialize a tensor with a uniform distribution within range [-1/sqrt(M), 1/sqrt(M)]
        tensor = torch.empty(*shape)
        r = 1 / math.sqrt(shape[0])
        torch.nn.init.uniform_(tensor, -r, r)
        return nn.Parameter(tensor, requires_grad=True)

In [ ]:
class PLRNN(PLRNN_Base):
    def __init__(self, M, N):
        super(PLRNN, self).__init__(M=M, P=M, N=N)

    def forward(self, z):
        #TODO: implement forward call (recursive equation z_t -> z_{t+1})
        pass

In [ ]:
class ALRNN(PLRNN_Base):
    def __init__(self, M, P, N):
        super(ALRNN, self).__init__(M, P, N)

    def forward(self, z):
        #TODO: implement forward call (recursive equation z_t -> z_{t+1})
        pass

## Utilities

In [ ]:
@torch.no_grad() # Disables gradient calculation to save memory and computation
def predict_free_sequence(model, x, T):
    """Predicts a sequence without updating model parameters (only for evaluation)"""
    
    b, N = x.size()

    Z = torch.empty(size=(T, b, model.M), device=x.device) # Initialize output tensor for the predicted sequence
    z = x @ model.B # Initialize first latent state
    z[:,0:N] = x

    # Predict sequence by passing previous state through the model
    for t in range(0, T):
        z = model(z)
        Z[t] = z

    return Z.permute(1, 0, 2)

In [ ]:
def predict_sequence_using_stf(model, x, forcing_interval):
    """Predicts a sequence using sparse teacher forcing (only for training)"""
    
    x_ = x.permute(1, 0, 2) # Permute input to shape (sequence_length, batch_size, feature_dim)
    T, b, N = x_.shape # T: sequence length, b: batch size, N: feature dimension
    Z = torch.empty(size=(T, b, model.M), device=x.device)
    
    z =  x_[0] @ model.B # Initialize first latent state
    z = teacher_force(z, x_[0]) # Apply teacher forcing to the initial state

    # Generate sequence predictions
    for t in range(0, T):
        # Apply teacher forcing at regular intervals
        if (t % forcing_interval == 0) and (t > 0):
            z = teacher_force(z, x_[t])
            
        # Update the latent state using the model
        z = model(z)
        Z[t] = z
        
    return Z.permute(1, 0, 2)

def teacher_force(z, x):
    # Teacher force the state z
    N = x.size(-1)
    z[:, :N] = x
    return z

In [ ]:
def plot_losses(losses):
    plt.rcParams["figure.figsize"] = (7,4)
    plt.rcParams.update({'font.size': 10})
    fig = plt.figure()

    ax = fig.add_subplot(111)
    ax.plot(losses,lw=3)
    ax.set_title('Training loss')
    ax.set_xlabel('epoch')
    ax.set_ylabel('loss')
    ax.set_yscale("log")

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_trajectory(X_gen, X_test, T_gen):
    Blues = plt.cm.Blues
    plt.rcParams["lines.linewidth"] = .35
    plt.rcParams["figure.figsize"] = (7,5)
    plt.rcParams["lines.linewidth"] = 2.
    plt.rcParams.update({'font.size': 10})
    
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    
    xs = X_gen[:, 0]
    ys = X_gen[:, 1]
    zs = X_gen[:, 2]
    
    ax.plot(X_test[:T_gen, 0], X_test[:T_gen, 1], X_test[:T_gen, 2], color=Blues(0.9), label="Ground Truth")
    ax.plot(xs, ys, zs, color=Blues(0.6), alpha=1., label="Freely Generated")
    
    plt.legend(loc="upper left")
    plt.axis("off")
    plt.show()

## Training routine

In [3]:
def train(model, dataset, learning_rate, num_epochs, forcing_interval, batches_per_epoch=50):
    model.train() # Set model to training mode
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.RAdam(model.parameters(), lr=learning_rate)
    losses = []
    
    with trange(num_epochs, desc="Training Progress") as epochs:
        # Loop over batches in each epoch
        for e in epochs:
            epoch_losses = []
            
            for _ in range(batches_per_epoch):
                optimizer.zero_grad() # Reset gradients for the optimizer
                
                #TODO: sample a batch of data from dataset (x: inputs, y: targets)

                #TODO: predict sequence using teacher forcing

                #TODO: calculate loss using loss_fn
                   
                loss.backward()
                optimizer.step()
                epoch_losses.append(loss.item())
            
            # Compute and store average loss for the epoch
            average_epoch_loss = sum(epoch_losses) / len(epoch_losses)
            epochs.set_postfix(loss=average_epoch_loss)
            losses.append(average_epoch_loss)
    
    return losses

## Load Data

In [5]:
X_train = np.load("lorenz63_train.npy").astype(np.float32)[500:]
X_test = np.load("lorenz63_test.npy").astype(np.float32)[500:]
T_train, N = X_train.shape
T_test = X_test.shape[0]

## Initialize Model

In [6]:
# total number of latent units
M = None #TODO
# number of piecewise linear units
P = None #TODO
# number readout units
N = X_train.shape[-1]

model = ALRNN(M=M, P=P, N=N)

## Training Hyperparameters

In [7]:
batch_size = None #TODO: set batch size
sequence_length = None #TODO: set sequence length
forcing_interval = None #TODO: set forcing interval
num_epochs = None #TODO: set number of epochs
learning_rate = 1e-3

## Initialize Dataset

In [ ]:
dataset = TimeSeriesDataset(X_train, sequence_length=sequence_length, batch_size=batch_size)

## Training

In [ ]:
losses = train(model, dataset, learning_rate, num_epochs, forcing_interval, batches_per_epoch=50)
plot_losses(losses)

## Saving/Loading

In [108]:
# torch.save(model.state_dict(), f"trained_model_M{M}_P{P}_N{N}.pt")

In [ ]:
# model = ALRNN(M=M, P=P, N=N)
# model.load_state_dict(torch.load("trained_model.pt"))

## Analysis

In [ ]:
X_test_torch=torch.tensor(X_test[:]).unsqueeze(0)

T_gen = 10000 # Predicted sequence length
T_r = 1000 # Transient cutoff length
orbit = predict_free_sequence(model, X_test_torch[:,0,:],T_gen+T_r).detach().numpy()[0][T_r:,:]
plot_trajectory(orbit, X_test, T_gen)